# Introduction to transformers: parallel vs sequential
Time one training step (forward and backward pass) of an LSTM layer and of a self-attention layer
(Keras `MultiHeadAttention` with one head) on the GPU, for sequences of growing length.
The LSTM must process the steps one after another; self-attention handles all positions at once.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import keras

keras.utils.set_random_seed(0)
print(tf.config.list_physical_devices("GPU"))

I0000 00:00:1791073381.173042 1202480 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## The two layers
Both map a batch of 16 sequences of 64-number vectors to sequences of the same shape.

In [2]:
d, batch = 64, 16
lstm = keras.layers.LSTM(d, return_sequences=True)
attn = keras.layers.MultiHeadAttention(num_heads=1, key_dim=d)


def make_step(layer, is_attention):
    @tf.function
    def step(x):
        with tf.GradientTape() as tape:
            y = layer(x, x) if is_attention else layer(x)
            loss = tf.reduce_sum(y)
        grads = tape.gradient(loss, layer.trainable_variables)
        return loss, grads
    return step


x = tf.random.normal((batch, 8, d))
lstm(x); attn(x, x)
print("LSTM parameters:", lstm.count_params(), "  attention parameters:", attn.count_params())

LSTM parameters: 33024   attention parameters: 16640


## Timing
For each length: one warm-up call (builds the graph), then 15 timed calls; we keep the fastest,
which is least disturbed by other programs sharing the GPU. The whole sweep is repeated 3 times
and the three fastest times are averaged.

In [3]:
lengths = [16, 32, 64, 128, 256, 512, 1024]
rows = []
for rep in range(3):
    for n in lengths:
        x = tf.random.normal((batch, n, d))
        for name, layer, is_attention in (("LSTM", lstm, False), ("self-attention", attn, True)):
            step = make_step(layer, is_attention)
            step(x)[0].numpy()                                   # warm-up
            times = []
            for _ in range(15):
                t0 = time.perf_counter()
                step(x)[0].numpy()
                times.append(time.perf_counter() - t0)
            rows.append(dict(rep=rep, length=n, layer=name, ms=1000 * min(times)))
t = pd.DataFrame(rows).groupby(["length", "layer"]).ms.mean().unstack().round(2)
t["sequential steps, LSTM"] = t.index
t["LSTM / attention"] = (t["LSTM"] / t["self-attention"]).round(1)
t.to_csv("data/timing.csv")
t

layer,LSTM,self-attention,"sequential steps, LSTM",LSTM / attention
length,,,,
16,1.60,0.55,16,2.9
32,2.43,1.09,32,2.2
64,2.84,0.92,64,3.1
128,5.34,1.38,128,3.9
256,6.28,1.19,256,5.3
512,18.26,2.91,512,6.3
1024,34.53,16.64,1024,2.1


How much does each time grow when the length grows 16 times, from 16 to 256, and from 64 to 1024?

In [4]:
for a, b in ((16, 256), (64, 1024)):
    print(f"{a} -> {b} words:  LSTM x{t.loc[b, 'LSTM'] / t.loc[a, 'LSTM']:.1f},  "
          f"self-attention x{t.loc[b, 'self-attention'] / t.loc[a, 'self-attention']:.1f}")

16 -> 256 words:  LSTM x3.9,  self-attention x2.2
64 -> 1024 words:  LSTM x12.2,  self-attention x18.1


The attention-weight matrix of one sequence has n x n entries: the work that grows with the square of the length.

In [5]:
for n in (256, 1024):
    print(f"n = {n}: {n * n:,} attention weights per sequence")

n = 256: 65,536 attention weights per sequence
n = 1024: 1,048,576 attention weights per sequence
